In [1]:
#@title 0) setup

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 1) GPT-2 Small From Scratch

### 1.1) Configuration

In [2]:
vocab_size = 50257
context_length = 1024
hidden_dimension = 768
number_of_heads = 12
number_of_layers = 12
feedforward_dimension = hidden_dimension * 4
dropout_rate = 0.1

### 1.2) Token Embedding + Positional Encoding

In [3]:
class ScratchTokenAndPositionEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_dimension)
        self.position_embedding = nn.Embedding(context_length, hidden_dimension)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, token_ids):
        batch_size, sequence_length = token_ids.shape
        token_embeddings = self.token_embedding(token_ids)
        positions = torch.arange(sequence_length, device=token_ids.device)
        position_embeddings = self.position_embedding(positions)
        return self.dropout(token_embeddings + position_embeddings)


### 1.3) Layer Normalization

In [4]:
class ScratchLayerNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.gain = nn.Parameter(torch.ones(hidden_dimension))
        self.bias = nn.Parameter(torch.zeros(hidden_dimension))
        self.epsilon = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(variance + self.epsilon)
        return self.gain * normalized + self.bias


### 1.4) GELU Activation

In [5]:
class ScratchGELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(
            math.sqrt(2.0 / math.pi) * (x + 0.044715 * x.pow(3))
        ))


### 1.5) Causal Multi-Head Attention

In [6]:
class ScratchCausalMultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.head_dimension = hidden_dimension // number_of_heads

        self.query_projection = nn.Linear(hidden_dimension, hidden_dimension)
        self.key_projection = nn.Linear(hidden_dimension, hidden_dimension)
        self.value_projection = nn.Linear(hidden_dimension, hidden_dimension)
        self.output_projection = nn.Linear(hidden_dimension, hidden_dimension)
        self.attention_dropout = nn.Dropout(dropout_rate)
        self.output_dropout = nn.Dropout(dropout_rate)

        self.register_buffer(
            "causal_mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape

        queries = self.query_projection(x).view(batch_size, sequence_length, number_of_heads, self.head_dimension).transpose(1, 2)
        keys = self.key_projection(x).view(batch_size, sequence_length, number_of_heads, self.head_dimension).transpose(1, 2)
        values = self.value_projection(x).view(batch_size, sequence_length, number_of_heads, self.head_dimension).transpose(1, 2)

        attention_scores = (queries @ keys.transpose(-2, -1)) / math.sqrt(self.head_dimension)
        attention_scores = attention_scores.masked_fill(
            self.causal_mask[:sequence_length, :sequence_length], float("-inf")
        )
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attention_dropout(attention_weights)

        attention_output = (attention_weights @ values).transpose(1, 2).contiguous().view(batch_size, sequence_length, hidden_dimension)
        return self.output_dropout(self.output_projection(attention_output))


### 1.6) Feed-Forward Network

In [7]:
class ScratchFeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.expand_linear = nn.Linear(hidden_dimension, feedforward_dimension)
        self.contract_linear = nn.Linear(feedforward_dimension, hidden_dimension)
        self.activation = ScratchGELU()
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = self.expand_linear(x)
        x = self.activation(x)
        x = self.contract_linear(x)
        return self.dropout(x)


### 1.7) Transformer Block

In [8]:
class ScratchTransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.first_layer_norm = ScratchLayerNorm()
        self.attention = ScratchCausalMultiHeadAttention()
        self.second_layer_norm = ScratchLayerNorm()
        self.feed_forward = ScratchFeedForward()

    def forward(self, x):
        x = x + self.attention(self.first_layer_norm(x))
        x = x + self.feed_forward(self.second_layer_norm(x))
        return x


### 1.8) GPT-2 Model

In [9]:
class ScratchGPT2(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = ScratchTokenAndPositionEmbedding()
        self.transformer_blocks = nn.ModuleList(
            [ScratchTransformerBlock() for _ in range(number_of_layers)])
        self.final_layer_norm = ScratchLayerNorm()
        self.unembedding = nn.Linear(hidden_dimension, vocab_size, bias=False)
        self.unembedding.weight = self.embedding.token_embedding.weight

    def forward(self, token_ids):
        x = self.embedding(token_ids)
        for block in self.transformer_blocks:
            x = block(x)
        x = self.final_layer_norm(x)
        logits = self.unembedding(x)
        return logits


### 1.9) Forward Pass

In [10]:
scratch_model = ScratchGPT2()
scratch_model.eval()

total_parameters = sum(p.numel() for p in scratch_model.parameters())
print(f"Total parameters: {total_parameters:,}")

sample_input = torch.randint(0, vocab_size, (1, 16))
with torch.no_grad():
    logits = scratch_model(sample_input)
print(f"Input shape:  {sample_input.shape}")
print(f"Output shape: {logits.shape}")


Total parameters: 124,439,808
Input shape:  torch.Size([1, 16])
Output shape: torch.Size([1, 16, 50257])


# 2) GPT-2 Small, PyTorch Components

### 2.1) Configuration

In [11]:
vocab_size = 50257
context_length = 1024
hidden_dimension = 768
number_of_heads = 12
number_of_layers = 12
feedforward_dimension = hidden_dimension * 4
dropout_rate = 0.1


### 2.2) Causal Multi-Head Attention

In [12]:
class PyTorchCausalMultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dimension,
            num_heads=number_of_heads,
            dropout=dropout_rate,
            batch_first=True,
        )
        self.register_buffer(
            "causal_mask",
            nn.Transformer.generate_square_subsequent_mask(context_length)
        )

    def forward(self, x):
        sequence_length = x.shape[1]
        mask = self.causal_mask[:sequence_length, :sequence_length]
        output, _ = self.attention(x, x, x, attn_mask=mask, is_causal=True)
        return output


### 2.3) Feed-Forward Network

In [13]:
class PyTorchFeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(hidden_dimension, feedforward_dimension),
            nn.GELU(approximate="tanh"),
            nn.Linear(feedforward_dimension, hidden_dimension),
            nn.Dropout(dropout_rate),
        )

    def forward(self, x):
        return self.network(x)


### 2.4) Transformer Block

In [14]:
class PyTorchTransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.first_layer_norm = nn.LayerNorm(hidden_dimension)
        self.attention = PyTorchCausalMultiHeadAttention()
        self.second_layer_norm = nn.LayerNorm(hidden_dimension)
        self.feed_forward = PyTorchFeedForward()

    def forward(self, x):
        x = x + self.attention(self.first_layer_norm(x))
        x = x + self.feed_forward(self.second_layer_norm(x))
        return x


### 2.5) GPT-2 Model

In [15]:
class PyTorchGPT2(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_dimension)
        self.position_embedding = nn.Embedding(context_length, hidden_dimension)
        self.dropout = nn.Dropout(dropout_rate)
        self.transformer_blocks = nn.ModuleList([PyTorchTransformerBlock() for _ in range(number_of_layers)])
        self.final_layer_norm = nn.LayerNorm(hidden_dimension)
        self.unembedding = nn.Linear(hidden_dimension, vocab_size, bias=False)
        self.unembedding.weight = self.token_embedding.weight

    def forward(self, token_ids):
        batch_size, sequence_length = token_ids.shape
        positions = torch.arange(sequence_length, device=token_ids.device)
        x = self.dropout(self.token_embedding(token_ids) + self.position_embedding(positions))
        for block in self.transformer_blocks:
            x = block(x)
        x = self.final_layer_norm(x)
        return self.unembedding(x)


### 2.6) Forward Pass

In [16]:
pytorch_model = PyTorchGPT2()
pytorch_model.eval()

total_parameters = sum(p.numel() for p in pytorch_model.parameters())
print(f"Total parameters: {total_parameters:,}")

sample_input = torch.randint(0, vocab_size, (1, 16))
with torch.no_grad():
    logits = pytorch_model(sample_input)
print(f"Input shape:  {sample_input.shape}")
print(f"Output shape: {logits.shape}")


Total parameters: 124,439,808
Input shape:  torch.Size([1, 16])
Output shape: torch.Size([1, 16, 50257])


# 3) Load Pretrained GPT-2 Weights

### 3.1) Load HuggingFace Weights into Scratch Model

In [17]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
hf_state = hf_model.state_dict()

def load_pretrained_weights(model):
    state = model.state_dict()

    state["embedding.token_embedding.weight"].copy_(hf_state["transformer.wte.weight"])
    state["embedding.position_embedding.weight"].copy_(hf_state["transformer.wpe.weight"])

    for layer_index in range(number_of_layers):
        hf_prefix = f"transformer.h.{layer_index}"

        state[f"transformer_blocks.{layer_index}.first_layer_norm.gain"].copy_(hf_state[f"{hf_prefix}.ln_1.weight"])
        state[f"transformer_blocks.{layer_index}.first_layer_norm.bias"].copy_(hf_state[f"{hf_prefix}.ln_1.bias"])

        qkv_weight = hf_state[f"{hf_prefix}.attn.c_attn.weight"]
        qkv_bias = hf_state[f"{hf_prefix}.attn.c_attn.bias"]
        q_weight, k_weight, v_weight = qkv_weight.split(hidden_dimension, dim=1)
        q_bias, k_bias, v_bias = qkv_bias.split(hidden_dimension, dim=0)

        # GPT-2 uses Conv1D (transposed) so we transpose into standard Linear layout
        state[f"transformer_blocks.{layer_index}.attention.query_projection.weight"].copy_(q_weight.t())
        state[f"transformer_blocks.{layer_index}.attention.query_projection.bias"].copy_(q_bias)
        state[f"transformer_blocks.{layer_index}.attention.key_projection.weight"].copy_(k_weight.t())
        state[f"transformer_blocks.{layer_index}.attention.key_projection.bias"].copy_(k_bias)
        state[f"transformer_blocks.{layer_index}.attention.value_projection.weight"].copy_(v_weight.t())
        state[f"transformer_blocks.{layer_index}.attention.value_projection.bias"].copy_(v_bias)

        state[f"transformer_blocks.{layer_index}.attention.output_projection.weight"].copy_(hf_state[f"{hf_prefix}.attn.c_proj.weight"].t())
        state[f"transformer_blocks.{layer_index}.attention.output_projection.bias"].copy_(hf_state[f"{hf_prefix}.attn.c_proj.bias"])

        state[f"transformer_blocks.{layer_index}.second_layer_norm.gain"].copy_(hf_state[f"{hf_prefix}.ln_2.weight"])
        state[f"transformer_blocks.{layer_index}.second_layer_norm.bias"].copy_(hf_state[f"{hf_prefix}.ln_2.bias"])

        state[f"transformer_blocks.{layer_index}.feed_forward.expand_linear.weight"].copy_(hf_state[f"{hf_prefix}.mlp.c_fc.weight"].t())
        state[f"transformer_blocks.{layer_index}.feed_forward.expand_linear.bias"].copy_(hf_state[f"{hf_prefix}.mlp.c_fc.bias"])
        state[f"transformer_blocks.{layer_index}.feed_forward.contract_linear.weight"].copy_(hf_state[f"{hf_prefix}.mlp.c_proj.weight"].t())
        state[f"transformer_blocks.{layer_index}.feed_forward.contract_linear.bias"].copy_(hf_state[f"{hf_prefix}.mlp.c_proj.bias"])

    state["final_layer_norm.gain"].copy_(hf_state["transformer.ln_f.weight"])
    state["final_layer_norm.bias"].copy_(hf_state["transformer.ln_f.bias"])

    model.load_state_dict(state)

pretrained_scratch_model = ScratchGPT2()
load_pretrained_weights(pretrained_scratch_model)
pretrained_scratch_model.eval()
print("Weights loaded successfully.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Weights loaded successfully.


### 3.2) Generate Text

In [28]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

def generate(model, prompt, max_new_tokens=40):
    token_ids = tokenizer.encode(prompt, return_tensors="pt")
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(token_ids[:, -context_length:])
        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        token_ids = torch.cat([token_ids, next_token], dim=1)
    return tokenizer.decode(token_ids[0])

print(generate(pretrained_scratch_model, "The meaning of life is"))


The meaning of life is not the same as the meaning of death.

The meaning of life is not the same as the meaning of death.

The meaning of life is not the same as the meaning of death


### 3.3) Verify Against HuggingFace

In [21]:
test_input = torch.randint(0, vocab_size, (1, 32))

with torch.no_grad():
    scratch_logits = pretrained_scratch_model(test_input)
    hf_logits = hf_model(test_input).logits

max_difference = (scratch_logits - hf_logits).abs().max().item()
print(f"Max absolute difference: {max_difference:.2e}")
print(f"Outputs match: {max_difference < 1e-4}")


Max absolute difference: 9.16e-05
Outputs match: True
